In [0]:
# notebook 3 - camada GOLD
# aqui a gente monta o star schema (dimensoes, bridges e fato), a tabela de contexto pra IA
# e responde as 6 perguntas de negocio com display()
# mesma ideia dos notebooks 1 e 2: tudo que e path/schema vem do medalhao_config
import importlib, sys
if 'medalhao_config' in sys.modules:
    del sys.modules['medalhao_config']
from medalhao_config import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(silver_schema, gold_schema)

medalhao_cinema.silver medalhao_cinema.gold


In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window


dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# leio as 6 tabelas da silver que a gold precisa (a cotacao ja foi aplicada na silver, entao nao preciso dela aqui)
# nomes das colunas que uso da silver (se na sua silver estiver diferente, ajusta so aqui):
#   tb_generos          -> id_filme, nome_genero
#   tb_pessoas_empresas -> id_filme, nome_entidade, tipo_entidade (Ator/Diretor/Roteirista/Produtora)
def ler_silver(nome):
    return spark.table(table(silver_schema, nome)).withColumn("id_filme", F.col("id_filme").cast("string"))

df_info       = ler_silver("tb_info_filmes")
df_fin        = ler_silver("tb_financeiro_filmes")
df_metricas   = ler_silver("tb_metricas_engajamento")
df_avaliacoes = ler_silver("tb_avaliacoes_usuarios")
df_generos    = ler_silver("tb_generos")
df_pessoas    = ler_silver("tb_pessoas_empresas")
# cria o schema e salva com o nome pedido na atividade
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

(df_silver_filmes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table(silver_schema, "tb_info_filmes")))

def salvar_gold(df, nome):
    # mesmo padrao de escrita da silver: delta + overwrite
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table(gold_schema, nome))
    print(f"gravado: {table(gold_schema, nome)} ({df.count()} linhas)")

In [0]:
# surrogate key com row_number (Window sem partitionBy = tudo numa particao so, ok pro tamanho dessa base)
w_sk_filme = Window.orderBy("id_filme")

dim_movies = (df_info
    .dropDuplicates(["id_filme"])                                    # garantia extra de 1 linha por filme
    .withColumn("sk_movie_id", F.row_number().over(w_sk_filme).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),                            # chave natural da origem
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)

dq_check("gold.dim_movies", "sk_movie_id não nulo", dim_movies, F.col("sk_movie_id").isNotNull())
dq_check_unique("gold.dim_movies", "sk_movie_id única", dim_movies, ["sk_movie_id"])
dq_check_unique("gold.dim_movies", "id_filme único", dim_movies, ["id_filme"])

salvar_gold(dim_movies, "dim_movies")
display(dim_movies.limit(10))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7272593530308674>, line 20
      2 w_sk_filme = Window.orderBy("id_filme")
      4 dim_movies = (df_info
      5     .dropDuplicates(["id_filme"])                                    # garantia extra de 1 linha por filme
      6     .withColumn("sk_movie_id", F.row_number().over(w_sk_filme).cast("bigint"))
   (...)     17     )
     18 )
---> 20 dq_check("gold.dim_movies", "sk_movie_id não nulo", dim_movies, F.col("sk_movie_id").isNotNull())
     21 dq_check_unique("gold.dim_movies", "sk_movie_id única", dim_movies, ["sk_movie_id"])
     22 dq_check_unique("gold.dim_movies", "id_filme único", dim_movies, ["id_filme"])

File <command-7272593530308708>, line 10, in dq_check(table_name, check_name, df, condition)
      8 def dq_check(table_name: str, check_name: str, df, condition):
      9     """Executa uma checagem de quali

In [0]:
dim_genres = (df_generos
    .select(F.trim(F.col("nome_genero")).alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

dq_check_unique("gold.dim_genres", "nome_genero único", dim_genres, ["nome_genero"])

salvar_gold(dim_genres, "dim_genres")
display(dim_genres)

In [0]:
dim_people = (df_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

dq_check("gold.dim_people", "tipo_pessoa só Ator/Diretor/Roteirista", dim_people,
         F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista"))
dq_check_unique("gold.dim_people", "nome+tipo único", dim_people, ["nome_pessoa", "tipo_pessoa"])

salvar_gold(dim_people, "dim_people")
display(dim_people.limit(10))

In [0]:
dim_companies = (df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

dq_check_unique("gold.dim_companies", "nome_produtora único", dim_companies, ["nome_produtora"])

salvar_gold(dim_companies, "dim_companies")
display(dim_companies.limit(10))

In [0]:
dim_reviews = (df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),     # conta todas as avaliacoes
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),  # avg ja ignora NULL
    )
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")  # so filmes que existem na dim
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dq_check_unique("gold.dim_reviews", "1 linha por filme", dim_reviews, ["sk_movie_id"])
dq_check("gold.dim_reviews", "nota média entre 0 e 10", dim_reviews,
         F.col("nota_media_usuarios").isNull() | F.col("nota_media_usuarios").between(0, 10))

salvar_gold(dim_reviews, "dim_reviews")
display(dim_reviews.limit(10))

## Tabelas ponte (bridge)

Um filme tem N generos / N pessoas / N produtoras. Se eu ligasse isso direto na fato, cada filme apareceria varias vezes e a soma de receita dobraria.
Por isso cada bridge guarda so o par de chaves (`sk_movie_id`, `sk_xxx_id`), 1 linha por par (`distinct()`).
Passo a passo: pego a tabela silver (id_filme + nome) -> join com `dim_movies` pra achar `sk_movie_id` -> join com a dimensao pra achar a outra sk.

In [0]:
# filme x genero
bridge_movie_genre = (df_generos
    .select("id_filme", F.trim(F.col("nome_genero")).alias("nome_genero"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

# filme x pessoa (join pelo nome E pelo tipo, porque a dim_people tem 1 linha por nome+tipo)
bridge_movie_person = (df_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("id_filme", F.trim(F.col("nome_entidade")).alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

# filme x produtora
bridge_movie_company = (df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

dq_check_unique("gold.bridge_movie_genre",   "par único", bridge_movie_genre,   ["sk_movie_id", "sk_genre_id"])
dq_check_unique("gold.bridge_movie_person",  "par único", bridge_movie_person,  ["sk_movie_id", "sk_person_id"])
dq_check_unique("gold.bridge_movie_company", "par único", bridge_movie_company, ["sk_movie_id", "sk_company_id"])

salvar_gold(bridge_movie_genre,   "bridge_movie_genre")
salvar_gold(bridge_movie_person,  "bridge_movie_person")
salvar_gold(bridge_movie_company, "bridge_movie_company")
display(bridge_movie_genre.limit(10))

## gold.fact_movies_performance

Grao = 1 filme. Parto da `dim_movies` (que ja e unica) e faco LEFT JOIN com financeiro e metricas.
Antes do join tiro duplicata por `id_filme` nas duas tabelas: se a silver tiver o mesmo filme 2x, o join dobraria a linha e quebraria o grao.
LEFT JOIN porque filme sem financeiro/metricas continua na fato (com NULL), nao pode sumir.

In [0]:
fin = df_fin.dropDuplicates(["id_filme"])
met = df_metricas.dropDuplicates(["id_filme"])

fact_movies_performance = (dim_movies.select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        "sk_movie_id",
        # financeiro (DECIMAL(18,2))
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        # engajamento (DOUBLE / INT)
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

# o teste mais importante da fato: nao pode ter mais linhas que filmes
dq_check_unique("gold.fact_movies_performance", "1 linha por filme (grão)", fact_movies_performance, ["sk_movie_id"])
print("filmes na dim:", dim_movies.count(), "| linhas na fato:", fact_movies_performance.count())

salvar_gold(fact_movies_performance, "fact_movies_performance")
display(fact_movies_performance.limit(10))

## gold.tb_contexto_filmes_ia (tabela pro time de IA)

1 linha por filme com tudo "achatado" em texto: generos, diretores, atores e produtoras juntos numa string, mais as metricas principais.
A coluna `texto_contexto` ja sai pronta pra virar embedding / prompt. `collect_set` tira repetidos e ignora NULL; `concat_ws` tambem ignora NULL.

In [0]:
generos_filme = (bridge_movie_genre.join(dim_genres, "sk_genre_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.sort_array(F.collect_set("nome_genero"))).alias("generos")))

pessoas_filme = (bridge_movie_person.join(dim_people, "sk_person_id")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Diretor",    F.col("nome_pessoa"))))).alias("diretores"),
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Roteirista", F.col("nome_pessoa"))))).alias("roteiristas"),
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Ator",       F.col("nome_pessoa"))))).alias("atores"),
    ))

produtoras_filme = (bridge_movie_company.join(dim_companies, "sk_company_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.sort_array(F.collect_set("nome_produtora"))).alias("produtoras")))

tb_contexto_filmes_ia = (dim_movies
    .join(fact_movies_performance.select("sk_movie_id", "receita_usd", "receita_brl", "lucro_usd", "popularidade", "nota_media_tmdb", "nota_media_imdb"), "sk_movie_id", "left")
    .join(dim_reviews.select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"), "sk_movie_id", "left")
    .join(generos_filme,    "sk_movie_id", "left")
    .join(pessoas_filme,    "sk_movie_id", "left")
    .join(produtoras_filme, "sk_movie_id", "left")
    .withColumn("texto_contexto", F.concat_ws(" | ",
        F.concat_ws(" ", F.lit("Filme:"),       F.col("titulo")),
        F.concat_ws(" ", F.lit("Ano:"),         F.col("ano_lancamento")),
        F.concat_ws(" ", F.lit("Status:"),      F.col("status_filme")),
        F.concat_ws(" ", F.lit("Gêneros:"),     F.col("generos")),
        F.concat_ws(" ", F.lit("Direção:"),     F.col("diretores")),
        F.concat_ws(" ", F.lit("Elenco:"),      F.col("atores")),
        F.concat_ws(" ", F.lit("Produtoras:"),  F.col("produtoras")),
        F.concat_ws(" ", F.lit("Nota TMDB:"),   F.col("nota_media_tmdb")),
        F.concat_ws(" ", F.lit("Nota IMDb:"),   F.col("nota_media_imdb")),
        F.concat_ws(" ", F.lit("Receita USD:"), F.col("receita_usd")),
        F.concat_ws(" ", F.lit("Sinopse:"),     F.col("sinopse")),
    ))
)

dq_check_unique("gold.tb_contexto_filmes_ia", "1 linha por filme", tb_contexto_filmes_ia, ["sk_movie_id"])

salvar_gold(tb_contexto_filmes_ia, "tb_contexto_filmes_ia")
display(tb_contexto_filmes_ia.select("titulo", "generos", "diretores", "produtoras", "texto_contexto").limit(5))

In [0]:
data_max = (filmes
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento"))
    .collect()[0][0])

print("lançamento realizado mais recente:", data_max)

def filmes_lancados_ultimos(anos):
    # janela: (data_max - N anos, data_max], so filmes lancados
    return filmes.filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") > F.add_months(F.lit(data_max), -12 * anos))
        & (F.col("data_lancamento") <= F.lit(data_max))
    )